导入edges+mapping for Node2Vec 中的networkx. 最后得到可以放入ml的graph embedding 版本特征csv


In [1]:
from pathlib import Path
import pandas as pd

In [6]:
# get input
EDGES_PATH = Path("../edges/edges_node2vec_L1A_elementId.csv")   # 你现在这份
MAP_PATH   = Path("../mappings/player_nodeid_map.csv")

edges = pd.read_csv(EDGES_PATH)
node_map = pd.read_csv(MAP_PATH)

print("edges shape:", edges.shape)
print("map shape  :", node_map.shape)

# 必要列检查
assert {"src","dst"}.issubset(edges.columns), "edges 必须至少有 src,dst"
assert {"node_id","player_id"}.issubset(node_map.columns), "map 必须有 node_id, player_id"

# 缺失检查
print("edges null rate:\n", edges[["src","dst"]].isna().mean())
print("map null rate:\n", node_map[["node_id","player_id"]].isna().mean())

# src 必须能对上 mapping（否则你无法对齐回 player_id）
missing_src = set(edges["src"]) - set(node_map["node_id"])
print("missing src in map:", len(missing_src))
assert len(missing_src) == 0, "edges 的 src 有一部分不在 player_nodeid_map 里，无法对齐"



edges shape: (1983, 3)
map shape  : (991, 2)
edges null rate:
 src    0.0
dst    0.0
dtype: float64
map null rate:
 node_id      0.0
player_id    0.0
dtype: float64
missing src in map: 0


In [ ]:
# 把csv里的边 变成一个图对象 G 后面可以让Node2Vec在上面做random walk
# 因为Node2Vec不认识df 只认识graph.  而NetworkX就是给你一个标准的图数据结构来存这些东西
import networkx as nx
# 去重
edges_simple = edges[["src","dst"]].dropna().drop_duplicates()

G = nx.Graph()  # Graph()是一个adjacency list的python 容器
G.add_edges_from(edges_simple.itertuples(index=False, name=None))

print("num_nodes:", G.number_of_nodes())
print("num_edges:", G.number_of_edges())

# sanity: 玩家节点覆盖率（mapping 里 player 的 node_id 有多少出现在图里）
player_nodes = set(node_map["node_id"])
covered_players = len(player_nodes & set(G.nodes()))
print("players covered in graph:", covered_players, "/", len(player_nodes))

num_nodes: 1375
num_edges: 1983
players covered in graph: 991 / 991


In [ ]:
# 训练Node2Vec 
from node2vec import Node2Vec

# 为了复现
SEED = 42

node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=20,
    num_walks=80,
    p=1.0,
    q=1.0,
    workers=4,
    seed=SEED
)

model = node2vec.fit(
    window=10,
    min_count=1,
    batch_words=128,
    seed=SEED
)

print("trained vocab size:", len(model.wv))